## Business Understanding

## Imports

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('wordnet')
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from nltk.stem import WordNetLemmatizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
from collections import Counter
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Data Understanding

In [2]:
df = pd.read_csv('../data/judge-1377884607_tweet_product_company.csv',encoding='latin1')
df.head()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


In [3]:
df.describe()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
count,9092,3291,9093
unique,9065,9,4
top,RT @mention Marissa Mayer: Google Will Connect...,iPad,No emotion toward brand or product
freq,5,946,5389


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9093 entries, 0 to 9092
Data columns (total 3 columns):
 #   Column                                              Non-Null Count  Dtype 
---  ------                                              --------------  ----- 
 0   tweet_text                                          9092 non-null   object
 1   emotion_in_tweet_is_directed_at                     3291 non-null   object
 2   is_there_an_emotion_directed_at_a_brand_or_product  9093 non-null   object
dtypes: object(3)
memory usage: 213.2+ KB


In [5]:
df.shape

(9093, 3)

## Data Preparation
1. **Lemmatization**- examines the morphology of the words and reduces each word to its most basic form or lemma eg studies become study.

2. **Lowercasing**- reduces the number of unique words words the model must handle, improves model consistency and ensures the model learns **all occurences of a word together**, and simplifies **text processing**

3. **Tokenization and removing stopwords**-splits the sentences into array of individual words. Stopwords(common words or pretty useless data) that contain litte or no information.

4. **Label Encoding**-a simple and very common technique in Machine Learning to convert **categorical labels**(text) into **numbers**.

5. **Stemming**- stemming refers to the removal of suffixes eg cats to cat.

In [6]:
df.isnull().sum() ## check for null entries



tweet_text                                               1
emotion_in_tweet_is_directed_at                       5802
is_there_an_emotion_directed_at_a_brand_or_product       0
dtype: int64

In [7]:
# dealing with the missing values. Do we drop of fill?
# for tweet, we will drop that one missing value

df = df.dropna(subset=['tweet_text']) # dropped the one missing tweet
df.isnull().sum() 

tweet_text                                               0
emotion_in_tweet_is_directed_at                       5801
is_there_an_emotion_directed_at_a_brand_or_product       0
dtype: int64

In [8]:
# Duplicates
duplicates = df['tweet_text'].duplicated().sum()  # check for duplicates in the 'tweet' column
print(f"Duplicate tweets: {duplicates}")
df = df.drop_duplicates(subset='tweet_text').reset_index(drop=True) # drop duplicates based on the 'tweet' column and reset index
print(f"Shape after removing duplicates: {df.shape}")

Duplicate tweets: 27
Shape after removing duplicates: (9065, 3)


In [9]:
## Normalizing sentiment labels and removing "I can't tell" entries
df['emotion_in_tweet_is_directed_at'] = df['emotion_in_tweet_is_directed_at'].str.lower().str.strip()
df = df[df['emotion_in_tweet_is_directed_at'] != "i can't tell"]

print(f"Shape after removing ambiguous labels: {df.shape}")
print(df['is_there_an_emotion_directed_at_a_brand_or_product'].value_counts())

Shape after removing ambiguous labels: (9065, 3)
is_there_an_emotion_directed_at_a_brand_or_product
No emotion toward brand or product    5372
Positive emotion                      2968
Negative emotion                       569
I can't tell                           156
Name: count, dtype: int64


## Text Cleaning Function

In [10]:
## Mapping the product names to their respective brands

apple_list = [x.lower() for x in ['iPad', 'Apple', 'iPad app', 'iPhone app', 'iPhone', 'Mac', 'MacBook']]
google_list = [x.lower() for x in ['Google' , 'Android App', 'Android']]

def map_to_brand(product_name):
    if pd.isna(product_name):
        return None
    
    product_name = str(product_name).lower()
    
    if product_name in apple_list:
        return 'Apple'
    elif product_name in google_list:
        return 'Google'
    else:
        return None

df['product'] = df['emotion_in_tweet_is_directed_at'].apply(map_to_brand)

def fill_missing_product(row):
    if pd.notna(row['product']):
        return row['product']
    
    text = str(row['tweet_text']).lower()
    
    if any(word in text for word in ['apple', 'ipad', 'iphone', 'mac']):
        return 'Apple'
    elif any(word in text for word in ['google', 'android', 'pixel']):
        return 'Google'
    else:
        return 'Not specified'

df['product'] = df.apply(fill_missing_product, axis=1)

In [11]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def get_clean_tokens(text):
    if not isinstance(text, str):
        return []
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove mentions and hashtag symbols (keep the word)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    # Remove punctuation and special characters
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize
    tokens = text.split()
    # Remove stop words
    
    tokens = [word for word in tokens if word not in stop_words]
    # Remove short tokens (1-2 chars)
    tokens = [t for t in tokens if len(t) > 2]
    # Lemmatize tokens
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

In [12]:
## Apply the cleaning function to the 'tweet_text' column and create a new column 'clean_text'

df['clean_text'] = df['tweet_text'].apply(lambda x: " ".join(get_clean_tokens(x)))

df[['tweet_text', 'clean_text']].head()

,tweet_text,clean_text
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iphone hr tweeting riseaustin dead need upgrad...
1,@jessedee Know about @fludapp ? Awesome iPad/i...,know awesome ipadiphone app youll likely appre...
2,@swonderlin Can not wait for #iPad 2 also. The...,wait ipad also sale sxsw
3,@sxsw I hope this year's festival isn't as cra...,hope year festival isnt crashy year iphone app...
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,great stuff fri sxsw marissa mayer google tim ...


In [13]:
positive_emotion = []
negative_emotion = []


for text, is_there_an_emotion_directed_at_a_brand_or_product in zip(df['tweet_text'], df['is_there_an_emotion_directed_at_a_brand_or_product']):
    tokens = get_clean_tokens(text)
    if is_there_an_emotion_directed_at_a_brand_or_product == 'Positive emotion':
        positive_emotion.extend(tokens)

    elif is_there_an_emotion_directed_at_a_brand_or_product == 'Negative emotion':
        negative_emotion.extend(tokens)

# Top 15 Positive tweets
print("\n=== Top 15 Words in POSITIVE Sentiment ===")
print(Counter(positive_emotion).most_common(15))

# Top 15 Negative tweets
print("\n=== Top 15 Words in NEGATIVE Sentiment ===")
print(Counter(negative_emotion).most_common(15))


=== Top 15 Words in POSITIVE Sentiment ===
[('sxsw', 3102), ('link', 1210), ('ipad', 1191), ('apple', 879), ('google', 690), ('store', 548), ('iphone', 522), ('app', 394), ('new', 359), ('austin', 290), ('popup', 218), ('android', 197), ('get', 181), ('amp', 178), ('launch', 174)]

=== Top 15 Words in NEGATIVE Sentiment ===
[('sxsw', 578), ('ipad', 194), ('iphone', 155), ('google', 141), ('apple', 107), ('link', 102), ('app', 60), ('store', 45), ('new', 43), ('like', 42), ('circle', 36), ('need', 35), ('social', 30), ('apps', 30), ('people', 29)]


## TF -IDF

TF - Maps out the frequency of tokens in our data
IDF - shows the relevance/Weight of words to our target variable

## Train/Test Split

In [14]:
X = df['clean_text']
y = df['is_there_an_emotion_directed_at_a_brand_or_product']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Modelling
The goal of this section is to build and compare classification models that can accurately predict sentiment from tweet text.

We begin with a baseline model, then introduce more complex models to improve performance.

## Baseline Model
### Logistic Regresison
Logistic Regression was chosen as a simple and interpretable linear model suitable for text classification.

In [15]:
### Baseline Model = Logistic Regresison with TF-IDF

# Building the full pipeline
pipeline_log= Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier',LogisticRegression())
])

#  training the baseline model
pipeline_log.fit(X_train,y_train)

# predictions
y_pred_baseline = pipeline_log.predict(X_test)

print(f'Baseline Logistic Regression with TFIDF')
print(classification_report(y_test,y_pred_baseline))

print(f'Confusion Matrix')
print(confusion_matrix(y_test,y_pred_baseline))

Baseline Logistic Regression with TFIDF
                                    precision    recall  f1-score   support

                      I can't tell       0.00      0.00      0.00        31
                  Negative emotion       0.69      0.08      0.14       114
No emotion toward brand or product       0.70      0.88      0.78      1074
                  Positive emotion       0.67      0.50      0.57       594

                          accuracy                           0.69      1813
                         macro avg       0.51      0.36      0.37      1813
                      weighted avg       0.68      0.69      0.66      1813

Confusion Matrix
[[  0   0  24   7]
 [  0   9  88  17]
 [  0   3 949 122]
 [  0   1 298 295]]


### Multinomial Naive Bayes 

In [16]:
# Alternatively, we can use Multinomial Naive Bayes and TFIDF
from sklearn.naive_bayes import MultinomialNB

# Building the full pipeline
nb_baseline_pipeline= Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('nb',MultinomialNB(alpha=0.5))
])

#  training the baseline model
nb_baseline_pipeline.fit(X_train,y_train)

# predictions
y_pred_nb_baseline = pipeline_log.predict(X_test)

print(f'Baseline Naive Bayes with TFIDF')
print(classification_report(y_test,y_pred_nb_baseline))

#Plotting the confusion matrix for the baseline model
print(f'Confusion Matrix')
print(confusion_matrix(y_test,y_pred_baseline))

Baseline Logistic Regression with TFIDF
                                    precision    recall  f1-score   support

                      I can't tell       0.00      0.00      0.00        31
                  Negative emotion       0.69      0.08      0.14       114
No emotion toward brand or product       0.70      0.88      0.78      1074
                  Positive emotion       0.67      0.50      0.57       594

                          accuracy                           0.69      1813
                         macro avg       0.51      0.36      0.37      1813
                      weighted avg       0.68      0.69      0.66      1813

Confusion Matrix
[[  0   0  24   7]
 [  0   9  88  17]
 [  0   3 949 122]
 [  0   1 298 295]]


In [17]:
### Tweaked Logistic Regresison with TF_-IDF


# Building the full pipeline
baseline_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=10000,
        ngram_range =(1,2), # to identify features like "love it", "not good". Helps with negation and sentiment phrase
        min_df=2 
    )),
    ('classifier',LogisticRegression(
        max_iter=1000,
        class_weight ='balanced',  # gives importance to rare classes eg negative emotion
        random_state=42,
        solver='lbfgs'
    ))
])

In [18]:
# Training our improved baseline model
baseline_pipeline.fit(X_train,y_train)

# Predicting and evaluating the model
y_pred_baseline = baseline_pipeline.predict(X_test)

print(f'Baseline Logistic Regression')
print(classification_report(y_test,y_pred_baseline))

# plotting confusion matrix for the improved baseline model
print(f'Confusion Matrix')
print(confusion_matrix(y_test,y_pred_baseline))

Baseline Logistic Regression
                                    precision    recall  f1-score   support

                      I can't tell       0.02      0.03      0.03        31
                  Negative emotion       0.36      0.66      0.46       114
No emotion toward brand or product       0.77      0.65      0.71      1074
                  Positive emotion       0.56      0.61      0.59       594

                          accuracy                           0.63      1813
                         macro avg       0.43      0.49      0.45      1813
                      weighted avg       0.66      0.63      0.64      1813

Confusion Matrix
[[  1   6  13  11]
 [  2  75  21  16]
 [ 28  87 700 259]
 [ 17  41 171 365]]


In [22]:
# Tweaked version of Naive Bayes with TF-IDF(for comparison)
nb_pipeline = Pipeline([
    ('tidf',TfidfVectorizer(
        max_features=10000,
        ngram_range = (1,2), 
        min_df=2
        )),

    ('nb',MultinomialNB(alpha=0.5))
])

#Training Naive Bayes
nb_pipeline.fit(X_train,y_train)

#Predictions
y_pred_nb= nb_pipeline.predict(X_test)

#printing clasification report for Naives Bayes
print(f'Naive Bayes')
print(classification_report(y_test, y_pred_nb))

# plotting confusion matrix for Naive Bayes
print(f'Confusion Matrix(Naive Bayes):')
print(confusion_matrix(y_test,y_pred_nb))

Naive Bayes
                                    precision    recall  f1-score   support

                      I can't tell       0.00      0.00      0.00        31
                  Negative emotion       0.91      0.09      0.16       114
No emotion toward brand or product       0.68      0.90      0.77      1074
                  Positive emotion       0.66      0.43      0.52       594

                          accuracy                           0.68      1813
                         macro avg       0.56      0.35      0.36      1813
                      weighted avg       0.68      0.68      0.64      1813

Confusion Matrix(Naive Bayes):
[[  0   0  27   4]
 [  0  10  87  17]
 [  0   1 964 109]
 [  0   0 337 257]]


## Evaluation

In [25]:
# Summary comparison of all models
